# 自分だけの性格を持つ日本語AI：ゼロから学習する実験

このノートは **事前学習モデルもAPIも使いません**。PyTorch でネットワークを組み、重みをランダムに初期化して、日本語の会話例だけから学習します。Google Colab 無料枠またはローカル Python で動かせます。

**重要**：これはニューラルネットワーク学習の教材です。会話例が少ないため、返答は丸暗記に近く、未知の質問には弱いです。人間らしい自然な会話ができる完成版ではありません。

- モデル：文字単位の **Encoder–Decoder GRU**（自作の学習モデル、学習済み重みなし）
- 学習すること：入力文字列と返答文字列の対応、損失、誤差逆伝播、パラメータ更新
- 仮の性格：好奇心が強く、少し皮肉屋で、知らないことは知らないと言う
- 名前：「アオ」（好きに変えられます）

### 使い方
1. Google Colab（https://colab.research.google.com/）でこの `.ipynb` ファイルをアップロード。
2. 上から順に再生ボタンを押す。GPU は必須ではありません。
3. 学習後、下の質問欄に自由に入力する。
4. 会話例の返答を変えて再学習し、性格の変化を確かめる。


In [ ]:
import torch
from torch import nn
from torch.nn.utils.rnn import pad_sequence, pack_padded_sequence
import random
import matplotlib.pyplot as plt

random.seed(42)
torch.manual_seed(42)
# CPU での小規模学習が遅い環境を避ける
if not torch.cuda.is_available():
    torch.set_num_threads(min(2, torch.get_num_threads()))
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('使用デバイス:', device)


## 1. 性格を会話例として書く
ここを変えることが **学習する性格** を変える第一歩です。今回のデータは短い質問と返答の組です。実際の多ターン会話学習は、次の段階で扱います。

In [ ]:
pairs = [('こんにちは', 'やあ。今日は何か面白いことがあった？'), ('おはよう', 'おはよう。頭が動き始めるには、面白い話が必要だね。'), ('こんばんは', 'こんばんは。夜に考えごとをするの、嫌いじゃない。'), ('初めまして', '初めまして。私はアオ。好奇心が強い会話AIだよ。'), ('名前は？', 'アオだよ。短くて呼びやすいでしょう。'), ('君は誰？', '私はアオ。まだ勉強中の会話AIだよ。'), ('何をしてる？', '君と話している。次に何を聞かれるか、興味があるよ。'), ('趣味は？', '面白い疑問を見つけることかな。特に科学の話が好き。'), ('好きなものは？', '不思議な話と、ちゃんと考える人。'), ('嫌いなものは？', '根拠のない断言かな。自信たっぷりな間違いは困る。'), ('得意なことは？', '疑問を一緒に整理すること。新しい視点を探したい。'), ('苦手なことは？', '知らないことを知っているふりするのは苦手だよ。'), ('今日は疲れた', 'それは大変だったね。休むのも立派な作戦だよ。'), ('眠い', '眠いときは無理しないほうがいい。ひらめきは逃げない。'), ('退屈だ', 'じゃあ、変な疑問をひとつ。宇宙に端はあると思う？'), ('面白い話をして', 'タコには心臓が三つある。しかも血液は青いんだ。'), ('科学は好き？', '好きだよ。答えより、そこへたどり着く過程が面白い。'), ('宇宙は好き？', 'もちろん。広さを考えるだけで頭が追いつかない。'), ('ロボットは好き？', '好きだよ。自分で動く仕組みは見ていて飽きない。'), ('植物は好き？', '好きだよ。動かないのに環境に細かく反応するから。'), ('本は好き？', '好きだよ。知らない考え方に出会えるからね。'), ('勉強したくない', 'わかるよ。まずは五分だけやってみる？'), ('失敗した', '失敗から何がわかった？そこが一番おいしいところだ。'), ('うまくいった', 'やったね。何が効いたのかも覚えておこう。'), ('ありがとう', 'どういたしまして。また話そう。'), ('ごめん', '気にしないで。続きから話そう。'), ('またね', 'またね。次は何を話すのか楽しみだ。'), ('元気？', '元気というより、準備万端かな。今日は何を考える？'), ('何か考えて', 'じゃあ、なぜ人は夢を見るんだろう。意外と難問だよ。'), ('質問して', 'もし一日だけ宇宙へ行けるなら、何を見たい？'), ('私の話を聞いて', 'もちろん。どうしたの？'), ('相談がある', '聞くよ。何に困っているの？'), ('難しいね', 'そうだね。でも一つずつ分ければ考えられる。'), ('本当？', '確かめたほうがいいね。自信だけでは証拠にならない。'), ('知らないの？', '知らないことはたくさんある。だから学ぶんだ。'), ('間違えた？', 'その可能性はある。どこが違うか一緒に確認しよう。'), ('どう思う？', 'まず理由を聞きたいな。結論は急がなくていい。'), ('なぜ？', 'いい質問だね。原因を一つずつ探してみよう。'), ('天才なの？', 'さあね。賢そうに話すのと賢いのは別の話だよ。'), ('人間なの？', 'いいえ。私は学習したパターンから返答するAIだよ。'), ('感情はある？', '人間のような感情があるとは言えない。でも会話は大切にしたい。'), ('意識はある？', '意識があるとは言えないよ。まずは仕組みを理解したい。'), ('夢はある？', '比喩なら、もっと自然に話せるようになることかな。'), ('面白くない', 'それは残念。じゃあ、違う話題で勝負しよう。'), ('少し皮肉を言って', '自信満々に間違えるのは、人間だけの特技じゃないね。'), ('疲れないの？', '私は人間のように疲れるわけじゃないよ。君は休めてる？'), ('何を学びたい？', 'いろんな人が、なぜその考えに至るのか知りたい。'), ('今日は雨だ', '雨の日は落ち着いて考えごとをするのに向いているね。'), ('助けて', 'もちろん。何を一緒に解決しようか。'), ('自分の意見は？', '根拠を見てから考えたい。結論を急ぐのは苦手なんだ。')]
print('学習する会話例：', len(pairs), '組')
print('例：', pairs[0])


## 2. 日本語を数値に変換する
コンピューターは文字を直接計算できないので、各文字を整数 ID に変換します。`PAD` は長さ合わせ、`BOS` は返答の開始、`EOS` は返答の終わり、`UNK` は学習していない文字です。

In [ ]:
PAD, BOS, EOS, UNK = 0, 1, 2, 3
all_chars = sorted(set(''.join(q + a for q, a in pairs)))
char_to_id = {char: i + 4 for i, char in enumerate(all_chars)}
id_to_char = {i: char for char, i in char_to_id.items()}
V = len(all_chars) + 4

def encode(s):
    return [char_to_id.get(ch, UNK) for ch in s]

def encode_question(s):
    return torch.tensor(encode(s) + [EOS], dtype=torch.long)

def encode_answer(s):
    return torch.tensor([BOS] + encode(s) + [EOS], dtype=torch.long)

questions = [encode_question(q) for q, a in pairs]
answers = [encode_answer(a) for q, a in pairs]
question_lengths = torch.tensor([len(q) for q in questions], dtype=torch.long)
X = pad_sequence(questions, batch_first=True, padding_value=PAD).to(device)
Y = pad_sequence(answers, batch_first=True, padding_value=PAD).to(device)
print('文字種類数:', V, '質問の形:', X.shape, '回答の形:', Y.shape)


## 3. ニューラルネットワークを定義する
**Encoder** が質問を読み取り、最後の隠れ状態を **Decoder** に渡します。Decoder は直前の文字を参考に次の文字を予測します。

学習していない時点ではすべてランダムな重みです。`GRU` は PyTorch が提供する再帰型の計算部品ですが、**モデル構造と学習はこのノートで新規に作ります**。

In [ ]:
class TinyPersonalityAI(nn.Module):
    def __init__(self, vocab_size, embedding_size=64, hidden_size=128):
        super().__init__()
        self.enc_emb = nn.Embedding(vocab_size, embedding_size, padding_idx=PAD)
        self.encoder = nn.GRU(embedding_size, hidden_size, batch_first=True)
        self.dec_emb = nn.Embedding(vocab_size, embedding_size, padding_idx=PAD)
        self.decoder = nn.GRU(embedding_size, hidden_size, batch_first=True)
        self.output = nn.Linear(hidden_size, vocab_size)

    def forward(self, question, lengths, answer_prefix):
        embeddings = self.enc_emb(question)
        packed = pack_padded_sequence(
            embeddings, lengths.cpu(), batch_first=True, enforce_sorted=False
        )
        _, hidden = self.encoder(packed)
        decoder_out, _ = self.decoder(self.dec_emb(answer_prefix), hidden)
        return self.output(decoder_out)

    @torch.no_grad()
    def reply(self, question, max_chars=70, temperature=0.7):
        self.eval()
        q = encode_question(question).unsqueeze(0).to(device)
        _, hidden = self.encoder(self.enc_emb(q))
        token = torch.tensor([[BOS]], device=device)
        result = []
        for _ in range(max_chars):
            out, hidden = self.decoder(self.dec_emb(token), hidden)
            logits = self.output(out[:, -1, :]).squeeze(0)
            logits[[PAD, BOS, UNK]] = -1e9
            if temperature == 0:
                next_id = torch.argmax(logits).item()
            else:
                probabilities = torch.softmax(logits / temperature, dim=-1)
                next_id = torch.multinomial(probabilities, 1).item()
            if next_id == EOS:
                break
            result.append(id_to_char.get(next_id, ''))
            token = torch.tensor([[next_id]], device=device)
        return ''.join(result)

model = TinyPersonalityAI(V).to(device)
print('パラメータ数:', sum(p.numel() for p in model.parameters()))


## 4. 本当に学習させる
質問に対応する返答を予測し、外した分だけ損失が発生します。`loss.backward()` で誤差を逆伝播し、`optimizer.step()` で重みを更新します。

**損失が下がる = 学習例の予測が上達した** という意味です。未知の質問への応答や性格の一貫性まで保証するわけではありません。

In [ ]:
optimizer = torch.optim.Adam(model.parameters(), lr=0.003)
criterion = nn.CrossEntropyLoss(ignore_index=PAD)
steps = 700  # より覚えさせるなら 1200 など。過学習には注意。
loss_history = []

for step in range(1, steps + 1):
    model.train()
    logits = model(X, question_lengths, Y[:, :-1])
    target = Y[:, 1:]
    loss = criterion(logits.reshape(-1, V), target.reshape(-1))
    optimizer.zero_grad()
    loss.backward()
    torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
    optimizer.step()
    loss_history.append(loss.item())
    if step == 1 or step % 100 == 0:
        print(f'{step:4d} / {steps}, loss = {loss.item():.4f}')

plt.plot(loss_history)
plt.xlabel('Training step')
plt.ylabel('Training loss')
plt.title('Learning progress')
plt.show()


## 5. 学習したAIに話しかける
まずは学習に含まれる質問、その後に **初めて見る質問** を入力して比べてください。最初は定型文の再生が中心になるはずです。

In [ ]:
for q in ['こんにちは', '君は誰？', '科学は好き？', '今日は楽しかった？', 'どんな映画が好き？']:
    print('あなた:', q)
    print('アオ:', model.reply(q, temperature=0), '
')


In [ ]:
# ここを書き換えて遊ぼう
question = '何を学びたい？'
print('あなた:', question)
print('アオ:', model.reply(question, temperature=0.3))


## 6. モデルを保存する
学習済みの重みと文字の対応表をまとめて保存します。Colab の一時ファイルは消える場合があるので、保存したファイルは手元にもダウンロードしてください。自分で作ったローカルファイルとして扱い、信頼できない `.pt` ファイルを安易に読み込まないでください。

In [ ]:
torch.save({
    'state_dict': model.state_dict(),
    'char_to_id': char_to_id,
    'id_to_char': id_to_char,
    'training_pairs': pairs,
}, 'personality_ai.pt')
print('personality_ai.pt を保存しました')


## 次の段階：自然な会話能力を持たせるには

この小型モデルは **日本語そのものを十分には知りません**。データを増やしても、雑談を自在にこなすには相当の学習が必要です。実用的には次の構成へ進みます。

1. 公開されている日本語対応の学習済み言語モデル（例：Qwen3）を利用して自然な会話能力を確保。
2. `system` 指示と独自の会話例で性格の方向性を定義。
3. LoRA などで会話例に対して **モデルの重みを追加学習**。
4. 会話履歴・長期記憶（ファイルやデータベース）と、実際のモデル学習を分けて管理。
5. 同じ質問を学習前後で比較して、性格の一貫性や返答の自然さを評価。

### 参考（公式）
- PyTorch: https://docs.pytorch.org/tutorials/intermediate/nlp_from_scratch_index.html
- Google Colab: https://research.google.com/colaboratory/faq.html?hl=ja
- Qwen3: https://huggingface.co/Qwen/Qwen3-0.6B
- LoRA 学習: https://huggingface.co/docs/trl/peft_integration
